# create_agent() 函数

`create_agent()` 是 LangChain 最核心的函数，它会创建一个完整的 Agent 图（StateGraph），包含模型调用、工具执行、循环控制等全部逻辑。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## 语法

```python
from langchain.agents import create_agent

agent = create_agent(
    model,                     # str | BaseChatModel：语言模型
    tools=None,                # Sequence：工具列表
    *,
    system_prompt=None,        # str | SystemMessage：系统提示
    middleware=(),             # Sequence[AgentMiddleware]：中间件列表
    response_format=None,      # ResponseFormat | type：结构化输出配置
    state_schema=None,         # type[AgentState]：自定义状态结构
    context_schema=None,       # type：运行时上下文结构
    checkpointer=None,         # Checkpointer：对话持久化
    store=None,                # BaseStore：跨会话存储
    interrupt_before=None,     # list[str]：在哪些节点前暂停
    interrupt_after=None,      # list[str]：在哪些节点后暂停
    debug=False,               # bool：是否输出详细日志
    name=None,                 # str：Agent 名称
    cache=None,                # BaseCache：缓存配置
)
```

### model 参数——模型配置

model 可以接受字符串（由 `init_chat_model()` 处理）或已构建好的 BaseChatModel 实例。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 方式 1：传字符串（最常用）
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    system_prompt="你是菜鸟教程 RUNOOB 的助手",
)

# 方式 2：传已构建好的模型实例
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0.3, max_tokens=500)
agent = create_agent(
    model=model,
    system_prompt="你是菜鸟教程 RUNOOB 的助手",
)

print("两种方式都支持")

> 推荐使用方式 1（传字符串），`create_agent()` 内部自动处理模型初始化、工具绑定等逻辑。

### tools 参数——工具列表

tools 接受三种格式：`@tool` 装饰的函数、Pydantic BaseModel、字典。

In [ ]:
from langchain.tools import tool

# 格式 1：@tool 装饰的函数（最常用）
@tool
def search_course(keyword: str) -> str:
    """搜索菜鸟教程课程"""
    return f"搜索结果：{keyword} 相关课程"

# 格式 2：Pydantic BaseModel
from pydantic import BaseModel, Field
class WeatherQuery(BaseModel):
    """查询天气"""
    city: str = Field(description="城市名称")

# 混合使用
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    tools=[search_course, WeatherQuery],
)

print(f"Agent 已创建，包含 {len(agent.tools)} 个工具")

### system_prompt 参数

定义 Agent 的行为角色和约束规则。支持字符串和 SystemMessage 对象。

In [ ]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage

# 方式 1：字符串
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    system_prompt="你是菜鸟教程 RUNOOB 的学习顾问。回答要简洁。",
)

# 方式 2：SystemMessage 对象
system_msg = SystemMessage(content="你是菜鸟教程 RUNOOB 的学习顾问。回答要简洁。")
agent = create_agent(model="deepseek:deepseek-v4-flash", system_prompt=system_msg)

print("两种方式等价")

### state_schema 参数——自定义状态

默认 AgentState 只包含 messages、jump_to 和 structured_response。通过扩展可添加业务字段。

In [ ]:
from typing import Annotated
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage
from langchain.tools import tool, InjectedState

class LearningAgentState(AgentState):
    """自定义状态，增加学习进度相关字段"""
    user_level: str
    completed_topics: list[str]

@tool
def track_progress(topic: str, state: Annotated[dict, InjectedState]) -> str:
    """记录用户的学习进度。

    Args:
        topic: 刚学完的主题名称
    """
    completed = state.get("completed_topics", [])
    completed.append(topic)
    return f"已完成 {len(completed)} 个主题：{', '.join(completed)}"

agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    tools=[track_progress],
    state_schema=LearningAgentState,
    system_prompt="你是菜鸟教程 RUNOOB 的学习助手。",
)

# result = agent.invoke({
#     "messages": [HumanMessage(content="我学完了 Python 基础")],
#     "user_level": "入门",
#     "completed_topics": ["HTML 基础"],
# })
# print(f"等级: {result.get('user_level')}")
# print(f"已完成: {result.get('completed_topics')}")

print("带自定义状态的 Agent 已创建")

### 返回值——CompiledStateGraph

`create_agent()` 返回一个 `CompiledStateGraph` 对象，提供多种运行方式：

| 方法 | 说明 | 适用场景 |
| --- | --- | --- |
| invoke(input, config) | 同步运行 | 脚本、简单接口 |
| ainvoke(input, config) | 异步运行 | Web 服务 |
| stream(input, config, stream_mode) | 同步流式 | 实时展示中间步骤 |
| astream(input, config, stream_mode) | 异步流式 | WebSocket、SSE |

**术语：**

- **CompiledStateGraph**：LangGraph 编译后的图，提供 invoke/stream 等方法
- **AgentState**：Agent 状态，包含 messages、jump_to、structured_response 等字段
- **state_schema**：自定义状态 Schema，扩展 AgentState